# **Libraries import**

In [236]:
import pandas as pd
import numpy as np
import datetime
import json
import requests

pd.set_option('display.max_columns', None)

# **Datasets loading**

In [237]:
RAW_BASE = 'https://raw.githubusercontent.com/YeHra/crm-unit-economics-ab-testing/refs/heads/main/data/raw/'

calls = pd.read_excel(RAW_BASE + 'calls_raw.xlsx', dtype={'Id': str, 'CONTACTID': str})
spend = pd.read_excel(RAW_BASE + 'spend_raw.xlsx')
deals = pd.read_excel(RAW_BASE + 'deals_raw.xlsx', dtype={'Id': str, 'Contact Name': str})
contacts = pd.read_excel(RAW_BASE + 'contacts_raw.xlsx', dtype={'Id': str})

# **Datasets processing**

## **Contacts**

In [238]:
contacts.head()

,Id,Contact Owner Name,Created Time,Modified Time
0,5805028000000645014,Rachel White,27.06.2023 11:28,22.12.2023 13:34
1,5805028000000872003,Charlie Davis,03.07.2023 11:31,21.05.2024 10:23
2,5805028000000889001,Bob Brown,02.07.2023 22:37,21.12.2023 13:17
3,5805028000000907006,Bob Brown,03.07.2023 05:44,29.12.2023 15:20
4,5805028000000939010,Nina Scott,04.07.2023 10:11,16.04.2024 16:14


###Datatype changes

In [239]:
contacts["Created Time"] = pd.to_datetime(contacts["Created Time"], errors="raise")
contacts["Modified Time"] = pd.to_datetime(contacts["Modified Time"], errors="raise")

/tmp/ipykernel_662/282674689.py:1: UserWarning: Parsing dates in %d.%m.%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  contacts["Created Time"] = pd.to_datetime(contacts["Created Time"], errors="raise")
/tmp/ipykernel_662/282674689.py:2: UserWarning: Parsing dates in %d.%m.%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  contacts["Modified Time"] = pd.to_datetime(contacts["Modified Time"], errors="raise")


###Drop duplicates

In [240]:
contacts.drop_duplicates(subset=contacts.columns[1:], inplace=True)

###Results control

In [241]:
contacts.isna().sum()

,0
Id,0
Contact Owner Name,0
Created Time,0
Modified Time,0


In [242]:
contacts.info()

<class 'pandas.core.frame.DataFrame'>
Index: 18510 entries, 0 to 18547
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Id                  18510 non-null  object        
 1   Contact Owner Name  18510 non-null  object        
 2   Created Time        18510 non-null  datetime64[ns]
 3   Modified Time       18510 non-null  datetime64[ns]
dtypes: datetime64[ns](2), object(2)
memory usage: 723.0+ KB


## **Calls**

In [243]:
calls.head()

,Id,Call Start Time,Call Owner Name,CONTACTID,Call Type,Call Duration (in seconds),Call Status,Dialled Number,Outgoing Call Status,Scheduled in CRM,Tag
0,5805028000000805001,30.06.2023 08:43,John Doe,NaN,Inbound,171.0,Received,NaN,NaN,NaN,NaN
1,5805028000000768006,30.06.2023 08:46,John Doe,NaN,Outbound,28.0,Attended Dialled,NaN,Completed,0.0,NaN
2,5805028000000764027,30.06.2023 08:59,John Doe,NaN,Outbound,24.0,Attended Dialled,NaN,Completed,0.0,NaN
3,5805028000000787003,30.06.2023 09:20,John Doe,5805028000000645014,Outbound,6.0,Attended Dialled,NaN,Completed,0.0,NaN
4,5805028000000768019,30.06.2023 09:30,John Doe,5805028000000645014,Outbound,11.0,Attended Dialled,NaN,Completed,0.0,NaN


###Datatype changes

In [244]:
calls['Call Start Time'] = pd.to_datetime(calls['Call Start Time'], errors='raise')

/tmp/ipykernel_662/1821417160.py:1: UserWarning: Parsing dates in %d.%m.%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  calls['Call Start Time'] = pd.to_datetime(calls['Call Start Time'], errors='raise')


###Drop duplicates

In [245]:
calls.drop_duplicates(subset=calls.columns[1:], inplace=True)
duplicate_mask_columns = [
    'Call Start Time', 'Call Owner Name', 'Call Duration (in seconds)',
    'Call Status'
]
calls.drop_duplicates(subset=duplicate_mask_columns, keep='first', inplace=True)

###Data transformation

In [246]:
calls['Call Duration (in min)'] = (calls['Call Duration (in seconds)'] / 60).round(2)

###Drop columns

In [247]:
calls = calls.drop(['Dialled Number', 'Tag', 'Call Duration (in seconds)'], axis=1)

###Fill NaN values

In [248]:
categories = [
    'Call Owner Name', 'Call Type', 'Call Status', 'Outgoing Call Status', 'CONTACTID']
for col in categories:
    calls[col] = calls[col].fillna('Unknown')
calls['Scheduled in CRM'] = (
    calls['Scheduled in CRM']
    .map({True: 'Yes', False: 'No'})
    .fillna('Unknown')
)

###Results control

In [249]:
calls.isna().sum()

,0
Id,0
Call Start Time,0
Call Owner Name,0
CONTACTID,0
Call Type,0
Call Status,0
Outgoing Call Status,0
Scheduled in CRM,0
Call Duration (in min),79


In [250]:
calls.info()

<class 'pandas.core.frame.DataFrame'>
Index: 92387 entries, 0 to 95873
Data columns (total 9 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   Id                      92387 non-null  object        
 1   Call Start Time         92387 non-null  datetime64[ns]
 2   Call Owner Name         92387 non-null  object        
 3   CONTACTID               92387 non-null  object        
 4   Call Type               92387 non-null  object        
 5   Call Status             92387 non-null  object        
 6   Outgoing Call Status    92387 non-null  object        
 7   Scheduled in CRM        92387 non-null  object        
 8   Call Duration (in min)  92308 non-null  float64       
dtypes: datetime64[ns](1), float64(1), object(7)
memory usage: 7.0+ MB


## **Spend**

In [251]:
spend.head()

,Date,Source,Campaign,Impressions,Spend,Clicks,AdGroup,Ad
0,2023-07-03,Google Ads,gen_analyst_DE,6,0.00,0,NaN,NaN
1,2023-07-03,Google Ads,performancemax_eng_DE,4,0.01,1,NaN,NaN
2,2023-07-03,Facebook Ads,NaN,0,0.00,0,NaN,NaN
3,2023-07-03,Google Ads,NaN,0,0.00,0,NaN,NaN
4,2023-07-03,CRM,NaN,0,0.00,0,NaN,NaN


###Datatype changes

In [252]:
spend["Date"] = pd.to_datetime(spend["Date"], errors="raise")

###Drop duplicates

In [253]:
spend.drop_duplicates(subset=spend.columns[1:], inplace=True)

###Data transformation

In [254]:
spend['Spend'] = spend['Spend'].replace(r'[€]', '', regex=True).astype(float)

In [255]:
spend = spend[spend['Source'] != 'Test']

In [256]:
spend = spend[
    (spend['Impressions'] != 0) &
    (spend['Spend'] != 0) &
    (spend['Clicks'] != 0)
]

###Fill NaN values

In [257]:
categories = ['Campaign', 'AdGroup', 'Ad']
for col in categories:
    spend[col] = spend[col].fillna('Unknown')

###Results control

In [258]:
spend.info()

<class 'pandas.core.frame.DataFrame'>
Index: 10134 entries, 1 to 20778
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   Date         10134 non-null  datetime64[ns]
 1   Source       10134 non-null  object        
 2   Campaign     10134 non-null  object        
 3   Impressions  10134 non-null  int64         
 4   Spend        10134 non-null  float64       
 5   Clicks       10134 non-null  int64         
 6   AdGroup      10134 non-null  object        
 7   Ad           10134 non-null  object        
dtypes: datetime64[ns](1), float64(1), int64(2), object(4)
memory usage: 712.5+ KB


In [259]:
spend.isna().sum()

,0
Date,0
Source,0
Campaign,0
Impressions,0
Spend,0
Clicks,0
AdGroup,0
Ad,0


## **Deals**

In [260]:
deals.head()

,Id,Deal Owner Name,Closing Date,Quality,Stage,Lost Reason,Page,Campaign,SLA,Content,Term,Source,Payment Type,Product,Education Type,Created Time,Course duration,Months of study,Initial Amount Paid,Offer Total Amount,Contact Name,City,Level of Deutsch
0,5805028000056864695,Ben Hall,NaN,NaN,New Lead,NaN,/eng/test,03.07.23women,NaN,v16,women,Facebook Ads,NaN,NaN,NaN,21.06.2024 15:30,NaN,NaN,NaN,NaN,5805028000056849495,NaN,NaN
1,5805028000056859489,Ulysses Adams,NaN,NaN,New Lead,NaN,/at-eng,NaN,NaN,NaN,NaN,Organic,NaN,Web Developer,Morning,21.06.2024 15:23,6.0,NaN,0,2000,5805028000056834471,NaN,NaN
2,5805028000056832357,Ulysses Adams,21.06.2024,D - Non Target,Lost,Non target,/at-eng,engwien_AT,00:26:43,b1-at,21_06_2024,Telegram posts,NaN,NaN,NaN,21.06.2024 14:45,NaN,NaN,NaN,NaN,5805028000056854421,NaN,NaN
3,5805028000056824246,Eva Kent,21.06.2024,E - Non Qualified,Lost,Invalid number,/eng,04.07.23recentlymoved_DE,01:00:04,bloggersvideo14com,recentlymoved,Facebook Ads,NaN,NaN,NaN,21.06.2024 13:32,NaN,NaN,NaN,NaN,5805028000056889351,NaN,NaN
4,5805028000056873292,Ben Hall,21.06.2024,D - Non Target,Lost,Non target,/eng,discovery_DE,00:53:12,website,NaN,Google Ads,NaN,NaN,NaN,21.06.2024 13:21,NaN,NaN,NaN,NaN,5805028000056876176,NaN,NaN


###Datatype changes

In [261]:
deals['Closing Date'] = pd.to_datetime(deals['Closing Date'], format='%d.%m.%Y',
                                       errors='raise')
deals['Created Time'] = pd.to_datetime(deals['Created Time'],
                                       format='%d.%m.%Y %H:%M',
                                       errors='raise')

###Drop duplicates

In [262]:
deals.drop_duplicates(subset=deals.columns[1:], inplace=True)

###Data transformation

In [263]:
mask = deals['Created Time'].dt.date > deals['Closing Date']
mask.sum()

np.int64(44)

In [264]:
deals.loc[mask, 'Created Time'] = deals.loc[mask, 'Closing Date']
deals.loc[mask, 'Closing Date'] = deals.loc[mask, 'Created Time'].dt.date

In [265]:
mask.sum()

np.int64(44)

In [266]:
def convert_to_minutes(x):
    """
    The function processes various time formats and converts them into a total
    number of minutes.
    """
    if pd.isna(x):
        return np.nan
    elif isinstance(x, datetime.time):
        return x.hour * 60 + x.minute + x.second / 60
    elif isinstance(x, datetime.timedelta):
        return x.total_seconds() / 60

deals['SLA Minutes'] = deals['SLA'].apply(convert_to_minutes).round(2)

In [267]:
def clean_currency_columns(df, columns_to_clean):
    """
    The function removes non-numeric characters from columns containing financial
    figures and converts the values to the float data type
    """
    for col in columns_to_clean:
        df[col] = (df[col]
            .replace(r'[€]', '', regex=True)
            .replace(r'\s+', '', regex=True)
            .replace(r'\.', '', regex=True)
            .replace(r',', '.', regex=True)
            .astype(float)
        )
    return df

columns_to_clean = ['Initial Amount Paid', 'Offer Total Amount']
deals = clean_currency_columns(deals, columns_to_clean)

In [268]:
mask = deals['Initial Amount Paid'] > deals['Offer Total Amount']
deals.loc[mask, 'Initial Amount Paid'] = deals.loc[mask, 'Offer Total Amount']
deals.loc[mask, 'Offer Total Amount'] = deals.loc[mask, 'Initial Amount Paid']

###Unnecessary values removing

In [269]:
deals.drop(columns=['SLA'], inplace=True)

In [270]:
deals = deals.replace('#REF!', pd.NA)
deals = deals.dropna(how='all')

In [271]:
deals = deals[deals['Created Time'] >= '2023-01-01']

In [272]:
deals['Product'].value_counts()

,count
Product,
Digital Marketing,1990
UX/UI Design,1022
Web Developer,575
Find yourself in IT,4
Data Analytics,1


In [273]:
deals = deals[~(deals['Product'].isin(['Find yourself in IT', 'Data Analytics']))]

In [274]:
deals['Page'].value_counts()

,count
Page,
/eng,5814
eng/digital-marketing,4548
/eng/test,2996
/workshop,1157
/webinar,1129
/,1079
/direct,1076
/eng/ux-ui,1058
/web-developer,658


In [275]:
#deals = deals[~(deals['Page'].isin(['/eng/test', '/test']))]

In [276]:
deals['Source'].value_counts()

,count
Source,
Facebook Ads,4848
Google Ads,4223
Organic,2586
Tiktok Ads,2051
SMM,1730
Youtube Ads,1657
CRM,1655
Bloggers,1089
Telegram posts,1001


In [277]:
deals = deals[deals['Source'] != 'Test']

In [278]:
deals['Lost Reason'].value_counts()

,count
Lost Reason,
Doesn't Answer,4118
Changed Decision,2133
Duplicate,1763
Non target,1739
Stopped Answering,1578
Invalid number,1472
needs time to think,652
Expensive,624
Conditions are not suitable,530


In [279]:
deals = deals[deals['Lost Reason'] != 'Duplicate']

###City processing

In [280]:
mode_values = deals.groupby('Contact Name')['City'].agg(
    lambda x: x.mode()[0] if not x.mode().empty else None
)
deals['City'] = deals['Contact Name'].map(mode_values)

In [281]:
deals['City'].unique().tolist()[:10]

[None,
 'Berlin',
 'Lahnstein',
 'Crailsheim',
 'Prenzlau',
 'Dortmund',
 'Stuttgart',
 'München',
 'Wien',
 'Offenbach am Main']

In [282]:
deals['City'] = deals['City'].replace('-', 'Unknown')

In [283]:
city_data = requests.get(RAW_BASE + 'city_data_google_en.json').json()

In [284]:
def get_city_info(city):
    """
    The function searches for a city in the global `city_data` dictionary and
    returns key geographical and administrative data as a pandas series.
    """
    info = city_data.get(city, {})
    return pd.Series({
        'longitude': info.get('longitude', None),
        'latitude': info.get('latitude', None),
        'country': info.get('country', None),
        'federal_state': info.get('federal_state', None),
        'city_en': info.get('city', None)
})

deals[['longitude', 'latitude', 'city_en', 'country', 'federal_state']] = deals['City'].apply(get_city_info)

###Level of Deutsch processing

In [285]:
mode_values = deals.groupby('Contact Name')['Level of Deutsch'].agg(
    lambda x: x.mode()[0] if not x.mode().empty else None
)
deals['Level of Deutsch'] = deals['Contact Name'].map(mode_values)

In [286]:
deals['Level of Deutsch'].unique().tolist()[:10]

[None, 'б1', 'в1', 'A2', 'в2', 'b1', 'В1', 'B1', 'в1-в2', 'А2 ( Б1 в июне)']

In [287]:
level_mapping = requests.get(RAW_BASE + 'deutsch_level_dict.json').json()

deals['Level of Deutsch'] = deals[
    'Level of Deutsch'].map(level_mapping).fillna('Unknown')

In [288]:
deals_category = deals.select_dtypes(include=['object']).columns[1:]
deals[deals_category] = deals[deals_category].fillna('Unknown')

In [289]:
deals.info()

<class 'pandas.core.frame.DataFrame'>
Index: 19656 entries, 0 to 21592
Data columns (total 28 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Id                   19656 non-null  object        
 1   Deal Owner Name      19656 non-null  object        
 2   Closing Date         13056 non-null  datetime64[ns]
 3   Quality              19656 non-null  object        
 4   Stage                19656 non-null  object        
 5   Lost Reason          19656 non-null  object        
 6   Page                 19656 non-null  object        
 7   Campaign             19656 non-null  object        
 8   Content              19656 non-null  object        
 9   Term                 19656 non-null  object        
 10  Source               19656 non-null  object        
 11  Payment Type         19656 non-null  object        
 12  Product              19656 non-null  object        
 13  Education Type       19656 non-null 

In [290]:
deals.isna().sum()

,0
Id,0
Deal Owner Name,0
Closing Date,6600
Quality,0
Stage,0
Lost Reason,0
Page,0
Campaign,0
Content,0
Term,0


# **Descriptive statistics**

##Calls statistic

In [291]:
calls_desc = calls[['Call Duration (in min)']].describe().T
calls_desc['mode'] = calls['Call Duration (in min)'].mode().iloc[0]
calls_desc

,count,mean,std,min,25%,50%,75%,max,mode
Call Duration (in min),92308.0,2.845796,6.789412,0.0,0.07,0.15,1.8,127.08,0.0


In [292]:
call_status = pd.DataFrame({
    'Count': calls['Call Status'].value_counts(),
    'Percentage': calls['Call Status'].value_counts(normalize=True) * 100
}).round(2).T

call_status

Call Status,Attended Dialled,Unattended Dialled,Missed,Received,Overdue,Scheduled Attended Delay,Cancelled,Scheduled Unattended Delay,Scheduled Attended,Scheduled Unattended,Scheduled
Count,69460.00,14015.00,5706.00,3070.00,57.00,20.00,19.00,17.00,14.00,6.00,3.0
Percentage,75.18,15.17,6.18,3.32,0.06,0.02,0.02,0.02,0.02,0.01,0.0


Die meisten Anrufe sind sehr kurz, meistens dauern sie 0 Minuten. Anhand der Prozentsätze der angenommenen und nicht angenommenen Anrufe lässt sich erkennen, dass die meisten Anrufe angenommen, aber schnell beendet werden. Eine weitere Analyse mit einer Gruppierung der Anrufe nach Kontakten ist erforderlich, um zu verstehen, ob nach den kurzen Anrufen längere Gespräche mit diesen Kontakten stattfanden.

##Spend statistic

In [293]:
spend_numeric = ['Impressions', 'Spend', 'Clicks']
spend_desc = spend[spend_numeric].describe().T
spend_desc['mode'] = spend[spend_numeric].mode().iloc[0].values
spend_desc

,count,mean,std,min,25%,50%,75%,max,mode
Impressions,10134.0,4971.223505,15997.428253,1.00,209.00,694.00,1930.25,431445.0,7.00
Spend,10134.0,13.778901,32.688924,0.01,2.06,5.81,11.64,774.0,4.25
Clicks,10134.0,40.517762,114.575542,1.00,3.00,9.00,22.00,2415.0,1.00


Der große Unterschied zwischen dem Medianwert und dem Durchschnittswert aller analysierten Indikatoren deutet auf große Schwankungen hin. Eine weitere Analyse mit einer Gruppierung nach Unternehmen und Werbequellen ist erforderlich.

##Deals statistic

In [294]:
deals_numeric = deals.select_dtypes(include=['number']).columns
deals_desc = deals[deals_numeric].describe().T
deals_desc['mode'] = deals[deals_numeric].mode().iloc[0].values
deals_desc

,count,mean,std,min,25%,50%,75%,max,mode
Course duration,3513.0,10.195844,1.837137,6.000000,11.000000,11.000000,11.000000,11.000000,11.000000
Months of study,835.0,5.449102,2.920733,0.000000,3.000000,5.000000,8.000000,11.000000,6.000000
Initial Amount Paid,4037.0,932.400297,1324.769932,0.000000,300.000000,1000.000000,1000.000000,11000.000000,1000.000000
Offer Total Amount,4056.0,7200.372041,4599.574809,0.000000,3500.000000,11000.000000,11000.000000,11500.000000,11000.000000
SLA Minutes,14856.0,1862.162170,12154.067312,0.050000,72.630000,326.720000,929.230000,448474.400000,10.180000
longitude,2674.0,10.428896,7.928819,-118.248870,8.154188,9.790653,12.096225,103.093976,13.404954
latitude,2674.0,50.668136,2.323569,7.880448,49.238170,50.989648,52.267791,59.931058,52.520007


Es ist ersichtlich, dass alle Indikatoren mit Ausnahme der Reaktionszeit relativ standardisiert sind. Längen- und Breitengrade weisen eine geringe Streuung auf, die sich auf Europa konzentriert. Der häufigste Wert entspricht Berlin.

Die SLA-Reaktionszeit weist eine enorme Streuung der Werte auf. Ersetzen wir 0,05 % der Maximalwerte durch die nächste Obergrenze

In [295]:
upper_bound = deals['SLA Minutes'].quantile(0.995)
deals.loc[deals['SLA Minutes'] > upper_bound, 'SLA Minutes'] = upper_bound

In [296]:
deals_numeric = deals.select_dtypes(include=['number']).columns
deals_desc = deals[deals_numeric].describe().T
deals_desc['mode'] = deals[deals_numeric].mode().iloc[0].values
deals_desc

,count,mean,std,min,25%,50%,75%,max,mode
Course duration,3513.0,10.195844,1.837137,6.000000,11.000000,11.000000,11.000000,11.000000,11.000000
Months of study,835.0,5.449102,2.920733,0.000000,3.000000,5.000000,8.000000,11.000000,6.000000
Initial Amount Paid,4037.0,932.400297,1324.769932,0.000000,300.000000,1000.000000,1000.000000,11000.000000,1000.000000
Offer Total Amount,4056.0,7200.372041,4599.574809,0.000000,3500.000000,11000.000000,11000.000000,11500.000000,11000.000000
SLA Minutes,14856.0,1380.545084,4725.058879,0.050000,72.630000,326.720000,929.230000,45266.203250,45266.203250
longitude,2674.0,10.428896,7.928819,-118.248870,8.154188,9.790653,12.096225,103.093976,13.404954
latitude,2674.0,50.668136,2.323569,7.880448,49.238170,50.989648,52.267791,59.931058,52.520007


Die Verarbeitung von 0,05 % der SLA-Minuten-Ausfälle hat die Verteilung verbessert. Für die weitere Analyse werden die verarbeiteten Werte verwendet.

In [297]:
cat_columns = ['Stage', 'Source', 'Quality', 'Product']
for col in cat_columns:
    deals_cat = pd.DataFrame({
    'Count': deals[col].value_counts(),
    'Percentage': deals[col].value_counts(normalize=True) * 100
}).round(2).T
    print(col)
    display(deals_cat)

Stage


Stage,Lost,Call Delayed,Registered on Webinar,Payment Done,Waiting For Payment,Qualificated,Registered on Offline Day,Need to Call - Sales,Need To Call,Test Sent,Need a consultation,New Lead,Free Education
Count,13915.00,2170.00,2066.00,852.00,323.00,128.00,85.00,32.00,31.00,25.00,23.00,5.00,1.00
Percentage,70.79,11.04,10.51,4.33,1.64,0.65,0.43,0.16,0.16,0.13,0.12,0.03,0.01


Source


Source,Facebook Ads,Google Ads,Tiktok Ads,SMM,Youtube Ads,Organic,CRM,Bloggers,Telegram posts,Webinar,Partnership,Offline
Count,4728.00,4114.00,2003.00,1669.00,1618.00,1496.00,1455.0,1074.00,993.00,303.00,201.00,2.00
Percentage,24.05,20.93,10.19,8.49,8.23,7.61,7.4,5.46,5.05,1.54,1.02,0.01


Quality


Quality,E - Non Qualified,D - Non Target,C - Low,Unknown,B - Medium,A - High
Count,6109.00,5975.0,3389.00,2231.00,1536.00,416.00
Percentage,31.08,30.4,17.24,11.35,7.81,2.12


Product


Product,Unknown,Digital Marketing,UX/UI Design,Web Developer
Count,16143.00,1944.00,1004.00,565.00
Percentage,82.13,9.89,5.11,2.87


Der Prozentsatz der verlorenen Transaktionen („Lost“, „Call Delayed“) ist mit 82 % sehr hoch. Die Konversionsrate in die Phase des erfolgreichen Abschlusses („Payment Done“) beträgt 4 %.

Dabei scheint die Anzahl der Transaktionen mit hoher und niedriger Qualität in quantitativer und prozentualer Hinsicht in dieser Phase keinen Zusammenhang mit der tatsächlichen Anzahl der abgeschlossenen und verlorenen Transaktionen zu haben.

Es ist eine zusätzliche Analyse der Manager erforderlich, um die Bedeutung des Parameters „Quality” zu bewerten.

Der Wert „Product” ist in 82 % der Fälle unbekannt, was möglicherweise mit der Anzahl der verlorenen Transaktionen („Lost”, „Call Delayed”) und derjenigen zusammenhängt, die sich in der Phase der Kundeninformation über die Produkte befinden („Registered on Webinar”). Von den bekannten Produkten ist „Digital Marketing” am beliebtesten. Eine zusätzliche Analyse der übrigen Produkte ist erforderlich.

Die wichtigsten Kanäle zur Kundengewinnung sind Facebook Ads (24 %) und Google Ads (21 %). Die Aktivität der übrigen Quellen ist um das Zweifache oder mehr geringer. Es ist eine Bewertung der Effektivität der Kanäle hinsichtlich der Kosten für die Gewinnung zahlungskräftiger Kunden und der Konversion in Zahlungen erforderlich.

# **Cleaned data saving**

In [298]:
for df, name in [(calls, 'calls'), (spend, 'spend'), (deals, 'deals'), (contacts, 'contacts')]:
    for col in df.select_dtypes(include='object').columns:
        types_found = df[col].apply(type).nunique()
        if types_found > 1:
            print(f'{name}.{col}: mixed types — {df[col].apply(type).value_counts().to_dict()}')

contacts.Contact Owner Name: mixed types — {<class 'str'>: 18509, <class 'bool'>: 1}


In [299]:
contacts[contacts['Contact Owner Name'].apply(lambda x: isinstance(x, bool))]

,Id,Contact Owner Name,Created Time,Modified Time
2197,5805028000008772190,False,2023-09-24 09:01:00,2023-10-13 16:44:00


In [300]:
contacts['Contact Owner Name'] = contacts['Contact Owner Name'].astype(str)

In [301]:
calls.to_parquet('calls_clean.parquet')
spend.to_parquet('spend_clean.parquet')
deals.to_parquet('deals_clean.parquet')
contacts.to_parquet('contacts_clean.parquet')

deals.to_excel('deals_clean.xlsx', index=False)
spend.to_excel('spend_clean.xlsx', index=False)